# Lesson 07 - Pattern di Progettazione Planning

Questo notebook dimostra il **Pattern di Progettazione Planning** per agenti AI utilizzando il Microsoft Agent Framework.
Imparerai come suddividere richieste di viaggio complesse in sottoattività strutturate, assegnarle ad agenti specialisti,
e eseguire il piano risultante — tutto utilizzando un output strutturato supportato da modelli Pydantic.


## Installazione


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Decomposizione del Compito

La decomposizione del compito è il nucleo del pattern di progettazione della pianificazione. Invece di chiedere a un singolo agente di gestire una richiesta complessa dall'inizio alla fine, dividiamo il problema in **sottocompiti** più piccoli e ben definiti. A ogni sottocompito viene assegnato un agente specialista (ad esempio, voli, hotel, attività) con priorità chiare e un ordine di dipendenza.

Questo approccio offre diversi vantaggi:
- **Chiarezza**: ogni sottocompito ha una responsabilità unica.
- **Parallelismo**: i sottocompiti indipendenti possono essere eseguiti contemporaneamente.
- **Affidabilità**: i fallimenti sono isolati ai singoli sottocompiti.
- **Monitoraggio del budget**: i costi sono stimati per ogni sottocompito e aggregati.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Creazione di un agente di pianificazione con output strutturato

L’agente di pianificazione agisce come un **coordinatore della reception**. Data una richiesta di viaggio ad alto livello, produce un `TravelPlan` strutturato — decomponendo la richiesta in sotto-compiti, impostando le priorità e identificando le dipendenze in modo che un concierge o un livello di esecuzione possa svolgere il lavoro.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Esecuzione di un piano con strumenti specialistici

Una volta che l'agente della reception ha prodotto un piano strutturato, l'**agente concierge** lo esegue.  
Ogni strumento specialistico gestisce una categoria di sottoattività (voli, hotel, attività). Il concierge itera attraverso le sottoattività del piano nell'ordine di dipendenza e assegna ciascuna allo strumento appropriato.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Riepilogo

In questa lezione hai imparato il **Pattern di Progettazione della Pianificazione** per agenti AI:

1. **Decomposizione del Compito** — Un agente di pianificazione alla reception suddivide una richiesta di viaggio complessa in sotto-compiti strutturati utilizzando modelli Pydantic, assegnando ciascuno a un agente specialista con priorità e dipendenze.
2. **Output Strutturato** — Tramite il passaggio di un `response_format` l’agente restituisce un oggetto `TravelPlan` validato invece di testo libero, rendendo l’elaborazione downstream affidabile.
3. **Esecuzione del Piano** — Un agente concierge itera attraverso i sotto-compiti usando strumenti specialistici (`book_flight`, `reserve_hotel`, `book_activity`) per eseguire il piano e riportarne i risultati.

Questo pattern separa *cosa fare* (pianificazione) da *come farlo* (esecuzione), rendendo gli agenti più modulari, testabili e facili da estendere.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Questo documento è stato tradotto utilizzando il servizio di traduzione automatica [Co-op Translator](https://github.com/Azure/co-op-translator). Pur impegnandoci per garantire precisione, si prega di considerare che le traduzioni automatiche possono contenere errori o imprecisioni. Il documento originale nella lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda la traduzione professionale effettuata da un umano. Non siamo responsabili per eventuali fraintendimenti o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
